# 05 — Prompt Patterns and Technique Selection

## Scenario
Northstar wants to automatically extract product codes from customer support emails. Product codes always follow the format `PRD-` followed by 4 digits (e.g., `PRD-1234`).

**The Rule of Minimum Complexity:** We will not start by building an agent, a RAG pipeline, or adding 50 examples. We will establish a baseline, measure the failure, and apply the *smallest* technique required to fix it.

In [ ]:
from pathlib import Path
from northstar.runtime import get_client
from lab05 import EVALUATION_SUITE, build_requests, deterministic_extract, render_worksheet, validate_code

client = get_client(Path("fixtures/replays.json"))
REQUESTS = {request.case_id: request for request in build_requests()}

def show(case_id):
    request = REQUESTS[case_id]
    print(f"\n=== {case_id} ===")
    print("SYSTEM:\n" + (request.system or "(none)"))
    for message in request.messages:
        print(f"{message.role.upper()}:\n{message.text}")
    response = client.generate(request)
    print("RECORDED RESPONSE:\n" + response.text)
    print("PARSED VALUE: exact text", repr(response.text))
    return request, response, response.text

## Phase 1: The Zero-Shot Baseline

We start with the simplest possible instruction.

In [ ]:
for case in EVALUATION_SUITE[:3]:
    _, _, observed = show(f"b05/zero/{case['id']}")
    print(case["id"], "expected:", case["expected"], "observed:", observed)
assert sum(
    (show(f"b05/zero/{case['id']}")[2].strip() == case["expected"])
    for case in EVALUATION_SUITE[:3]
) == 1

## Phase 2: Applying a System Instruction Constraint

**Hypothesis:** The model is prioritizing a conversational persona. If we apply a strict system instruction to constrain the output format, we can eliminate the filler.
**Technique:** System Instruction.

In [ ]:
for case in EVALUATION_SUITE[:3]:
    _, _, observed = show(f"b05/system/{case['id']}")
    print(case["id"], "expected:", case["expected"], "observed:", observed)
assert show("b05/system/multiple_numbers")[2].strip() == "88412"

## Phase 3: Applying Few-Shot Examples for a Decision Boundary

**Hypothesis:** The model doesn't fully understand the specific pattern of a 'product code' vs an 'order number'. 
**Technique:** Few-Shot Examples (specifically targeting the boundary case).

In [ ]:
for case in EVALUATION_SUITE:
    _, _, observed = show(f"b05/few/{case['id']}")
    assert validate_code(observed.strip()) == case["expected"] or (
        case["expected"] == "NONE" and observed.strip() == "NONE"
    )
print("Few-shot succeeds on the formatted variant:", show("b05/few/formatted_variant")[2])

## Phase 4: Compare deterministic code

The requirement contains an exact canonical pattern, so we must measure a zero-model-token parser before accepting additional prompt complexity.

In [ ]:
deterministic_observed = [deterministic_extract(case["message"]) for case in EVALUATION_SUITE]
deterministic_correct = sum(
    actual == case["expected"]
    for actual, case in zip(deterministic_observed, EVALUATION_SUITE)
)
print("Deterministic extraction:", list(zip([case["expected"] for case in EVALUATION_SUITE], deterministic_observed)))
assert deterministic_correct == 3
assert deterministic_observed[-1] == "NONE"

## Conclusion

By measuring failures first, we avoided building a massive prompt. We added exactly two techniques: a System Instruction to enforce constraints, and a single Few-Shot example to teach a specific boundary.

## Takeaway

The assertions above describe the deterministic recorded run. Change one prompt variable, rerun the lab, and measure the trade-off.

## References

See the course README for the folded reference material and links.

## Reading the technique-selection experiment

The frozen suite contains a straightforward product code, a message with no
code, a message containing both an order number and a product code, and a
formatted variant with lowercase letters and a space. The first three cases are
the original evaluation set; the fourth is held out as a boundary probe for the
conclusion. Keeping the formatted variant separate prevents silently changing
the headline score while still testing generalization.

Zero-shot establishes the baseline and records conversational filler or a wrong
number in the replay. A system instruction narrows the output vocabulary and
improves the recorded score, but the `multiple_numbers` response still selects
88412. Few-shot examples make the decision boundary explicit and succeed on all
four recorded cases. This is a measured result for this small suite, not a
promise that examples solve every extraction problem.

The final phase compares a deterministic validator with prompting. The exact
regular expression consumes zero model tokens and scores 3/4 on the full
suite. It intentionally rejects `prd 9921` rather than silently
normalizing it. That failure is useful: code is preferable when the rule is
exact, while a model or a separate normalization step may be appropriate when
language variation is part of the requirement. The worksheet turns this into a
repeatable choice: identify the failure, choose the smallest technique, and
measure the result before adding complexity.